# Fine-tune OpenTSLM on Amazon SageMaker

Fine-tune the pretrained OpenTSLM model ([opentime/tslm-nov26](https://huggingface.co/opentime/tslm-nov26) on Hugging Face) on ARFBench as a SageMaker training job, then deploy the fine-tuned model to a real-time endpoint and ask it about a time series.

Run it on your machine with AWS credentials for the account and region to use (for example after `aws sso login`, with `AWS_PROFILE` and `AWS_DEFAULT_REGION` set), or in SageMaker Studio. Use a Python 3.12 kernel: the training job runs your fine-tuning function in the SageMaker PyTorch 2.9 image, which uses Python 3.12.

Keep `utils.py` next to this notebook. It holds the OpenTSLM model, its fine-tuning code and the endpoint's request handler. The training job and the endpoint both import it.

Fine-tuning runs on one `ml.g7e.2xlarge` instance and the endpoint on one `ml.g6.2xlarge`. We ran it in eu-north-1; check that your region offers both types. New accounts often have a quota of 0 for both. Before you start, request at least 1 for "ml.g7e.2xlarge for training job usage" and "ml.g6.2xlarge for endpoint usage" in Service Quotas.

## Set up

The next cell finds or creates what the notebook needs in your account: the SageMaker default bucket (`sagemaker-<region>-<account ID>`) and an execution role. In SageMaker Studio it uses the Studio execution role. Elsewhere it creates the role `OpenTSLMSageMakerRole` once, with the AWS managed policy `AmazonSageMakerFullAccess`, so your credentials need permission to create IAM roles the first time.

In [ ]:
%pip install --quiet "sagemaker>=3.29,<4" "botocore[crt]"  # crt: for profiles that use aws login

In [ ]:
import json
import os
import time
import uuid

from sagemaker.core.helper.session_helper import Session, get_execution_role

session = Session()
REGION = session.boto_region_name
BUCKET = session.default_bucket()  # created if it does not exist


def execution_role(name="OpenTSLMSageMakerRole"):
    if os.path.exists("/opt/ml/metadata/resource-metadata.json"):  # SageMaker Studio
        return get_execution_role(session)
    iam = session.boto_session.client("iam")
    try:
        return iam.get_role(RoleName=name)["Role"]["Arn"]
    except iam.exceptions.NoSuchEntityException:
        pass
    trust = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "sagemaker.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }
    arn = iam.create_role(RoleName=name, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    iam.attach_role_policy(
        RoleName=name, PolicyArn="arn:aws:iam::aws:policy/AmazonSageMakerFullAccess"
    )
    time.sleep(15)  # SageMaker can assume a new role only after a few seconds
    return arn


ROLE_ARN = execution_role()
unique_id = uuid.uuid4().hex[:8]
finetuned_endpoint_name = f"opentslm-arfbench-{unique_id}"
print(f"Region {REGION}, bucket {BUCKET}, role {ROLE_ARN}")

## Fine-tune on ARFBench

ARFBench is a public benchmark of multiple-choice questions about anomalies in Datadog's software-incident telemetry. TimeNet serves it as a versioned dataset of tasks: each question with its options, its answer, and the metric series it refers to.

The test set holds out 20% of the questions, split by incident so that no incident appears in both sets. `split_by_incident` in `utils.py` computes this split.

The `@remote` decorator runs `fine_tune` as a SageMaker training job on one GPU and returns its result. It sends the function and `utils.py` from the notebook folder to the job.

In [ ]:
from importlib.metadata import version

from sagemaker.core import image_uris
from sagemaker.core.remote_function import remote

# The pretrained model to fine-tune: a Hugging Face repository (hf://ORG/REPO) or a local model
# directory. The repository can get newer checkpoints over time, so set this to the checkpoint
# that fits your use case.
BASE_MODEL = "hf://opentime/tslm-nov26"
FINETUNED_PREFIX = f"s3://{BUCKET}/opentslm-arfbench/{unique_id}"
# The job deserializes the function with sagemaker-core, so it needs the version of this kernel.
SAGEMAKER_CORE = f"sagemaker-core=={version('sagemaker-core')}"
# 1 x NVIDIA RTX PRO 6000 (96 GB). ml.g7e.4xlarge and ml.g7e.8xlarge have the same GPU: use one
# of them if the job waits for capacity. Fine-tuning needs most of the 96 GB.
TRAINING_INSTANCE_TYPE = "ml.g7e.2xlarge"


@remote(
    instance_type=TRAINING_INSTANCE_TYPE,
    image_uri=image_uris.retrieve(
        "pytorch",
        region=REGION,
        version="2.9.0",
        py_version="py312",
        instance_type=TRAINING_INSTANCE_TYPE,
        image_scope="training",
    ),
    role=ROLE_ARN,
    s3_root_uri=f"s3://{BUCKET}/opentslm-remote",
    include_local_workdir=True,  # utils.py
    pre_execution_commands=[
        f"pip install --quiet {SAGEMAKER_CORE} transformers==4.57.6 safetensors timenet==0.12.0"
    ],
    environment_variables={"HF_HOME": "/opt/ml/input/huggingface"},
    volume_size=100,
)
def fine_tune(base_model, finetuned_model):
    from utils import OpenTSLM, split_by_incident
    from timenet.client import TimeNet

    dataset = TimeNet().load("datadog/arfbench")  # the hosted TimeNet registry
    train, test = split_by_incident(dataset.tasks, test_fraction=0.2, seed=0)

    model = OpenTSLM.load(base_model)
    before = model.eval(test, metrics=["accuracy"])
    model.fine_tune(train, epochs=3)
    after = model.eval(test, metrics=["accuracy"])
    model.save(finetuned_model)  # the files of a Hugging Face model repository
    return before["accuracy"], after["accuracy"]


before, after = fine_tune(BASE_MODEL, FINETUNED_PREFIX)
print(f"Test accuracy before fine-tuning: {before:.1%}")
print(f"Test accuracy after fine-tuning:  {after:.1%}")

## Deploy the fine-tuned model

The SageMaker PyTorch inference image serves an S3 prefix that holds the model files and a `code/` folder. The next cell puts three files in that folder: `utils.py`, `inference.py`, which takes the request handler from `utils.py`, and `requirements.txt`, which installs Transformers when the endpoint starts. OpenTSLM needs one GPU with at least 24 GB of memory and bfloat16 support, such as `ml.g6.2xlarge` (1 x NVIDIA L4).

A request holds up to 4 questions. Each question has up to 32 series and the question text; a series has its `values` (`None` marks a missing value), a short `name`, and a `text` that describes it. The response holds one answer per question.

In [ ]:
from sagemaker.core.s3 import S3Uploader
from sagemaker.serve.model_builder import ModelBuilder

INFERENCE_INSTANCE_TYPE = "ml.g6.2xlarge"

S3Uploader.upload("utils.py", f"{FINETUNED_PREFIX}/code", sagemaker_session=session)
for name, body in [
    ("inference.py", "from utils import input_fn, model_fn, output_fn, predict_fn\n"),
    ("requirements.txt", "transformers==4.57.6\n"),  # the image has torch and numpy
]:
    S3Uploader.upload_string_as_file_body(
        body, f"{FINETUNED_PREFIX}/code/{name}", sagemaker_session=session
    )

finetuned_builder = ModelBuilder(
    image_uri=image_uris.retrieve(
        "pytorch",
        region=REGION,
        version="2.6.0",
        py_version="py312",
        instance_type=INFERENCE_INSTANCE_TYPE,
        image_scope="inference",
    ),
    env_vars={
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        "SAGEMAKER_MODEL_SERVER_WORKERS": "1",  # one copy of the model per GPU
        "TS_DEFAULT_STARTUP_TIMEOUT": "1800",  # loading the model takes several minutes
        "SAGEMAKER_TS_RESPONSE_TIMEOUT": "300",
    },
    s3_model_data_url={
        "S3DataSource": {
            "S3Uri": f"{FINETUNED_PREFIX}/",
            "S3DataType": "S3Prefix",
            "CompressionType": "None",
        }
    },
    role_arn=ROLE_ARN,
    instance_type=INFERENCE_INSTANCE_TYPE,
)
finetuned_model = finetuned_builder.build(model_name=finetuned_endpoint_name)
finetuned_endpoint = finetuned_builder.deploy(
    endpoint_name=finetuned_endpoint_name,
    model_data_download_timeout=1800,  # the weights take 14 GB
    container_startup_health_check_timeout=1800,
)


In [ ]:
def prompt(endpoint, series, question):
    request = {
        "instances": [{"series": series, "question": question}],
        "parameters": {"max_new_tokens": 128},
    }
    response = endpoint.invoke(body=json.dumps(request), content_type="application/json")
    return json.loads(response.body.read())["generated_text"][0]


# One hour of CPU utilization, one sample per minute, with a five-minute spike at minute 28.
cpu = [21, 20, 22, 19] * 7 + [78, 85, 82, 80, 79] + [22, 20, 21] * 9
cpu_series = [
    {"values": cpu, "name": "CPU utilization (%)", "text": "CPU utilization (%), 1 sample/minute:"}
]
question = "Is there an anomaly in this series? If so, when does it start and end?"
print(prompt(finetuned_endpoint, cpu_series, question))

## Clean up

The next cell deletes the endpoint, its configuration and the SageMaker model, also after a failed deploy. The fine-tuned model files stay under `FINETUNED_PREFIX` in your bucket, and the role `OpenTSLMSageMakerRole` stays for the next run.

In [ ]:
sagemaker_client = session.boto_session.client("sagemaker")
try:
    config = sagemaker_client.describe_endpoint(EndpointName=finetuned_endpoint_name)
    sagemaker_client.delete_endpoint(EndpointName=finetuned_endpoint_name)
    sagemaker_client.delete_endpoint_config(EndpointConfigName=config["EndpointConfigName"])
except sagemaker_client.exceptions.ClientError as error:
    print(error)  # no endpoint: the deploy did not get that far
try:
    sagemaker_client.delete_model(ModelName=finetuned_endpoint_name)
except sagemaker_client.exceptions.ClientError as error:
    print(error)